https://gist.github.com/waveletdeboshir/8bf52f04bf78018194f25b2390c08309

In [2]:
import torch
torch.cuda.is_available()

ImportError: libnvshmem_host.so.3: cannot open shared object file: No such file or directory

In [3]:
import os
from pathlib import Path
from IPython.display import Audio

from transformers import pipeline
import torch

from my_audio import MyAudio


ImportError: libnvshmem_host.so.3: cannot open shared object file: No such file or directory

In [ ]:
torch.cuda.empty_cache()
print(torch.cuda.memory_summary(device=None, abbreviated=False))

In [ ]:
audio_path = Path("dataset/kish_durak_i_molniya.mp3")
assert audio_path.exists(), f"File not found: {audio_path}"

audio_path

In [ ]:
my_audio = MyAudio()
my_audio.load_audio(audio_path)

In [ ]:
whisper_pipe = pipeline(
    "automatic-speech-recognition",
    model="openai/whisper-large-v3",
)

In [ ]:
from pyannote.audio import Pipeline

diarization_pipe = Pipeline.from_pretrained(
    "pyannote/speaker-diarization@2.1", use_auth_token=True
)


In [ ]:
torch.cuda.empty_cache()
print(torch.cuda.memory_summary(device=None, abbreviated=False))

whisper_result = whisper_pipe(
    str(audio_path),
    chunk_length_s=30,
    batch_size=8,
    return_timestamps=True,
    generate_kwargs={"language": "russian", "task": "transcribe"},
)

In [ ]:
whisper_result = whisper_pipe(
    str(audio_path),
    return_timestamps=True,
    generate_kwargs={"language": "russian", "task": "transcribe"},
)


In [ ]:
whisper_text = whisper_result["text"].strip()

print("Whisper ASR:\n")
print(whisper_text)

whisper_result

# start with diarization

In [ ]:
input_tensor = torch.from_numpy(my_audio.audio).float()
outputs = diarization_pipe(
    {"waveform": input_tensor, "sample_rate": my_audio.sample_rate}
)

outputs.for_json()["content"]


In [ ]:
from speechbox import ASRDiarizationPipeline

asrd_pipeline = ASRDiarizationPipeline(
    asr_pipeline=whisper_pipe, diarization_pipeline=diarization_pipe
)
